In [2]:
!pip install selenium webdriver-manager requests pandas

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 2.2 MB/s  0:00:04m0:00:0100:01
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: certifim━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [urllib3]
    Found existing installation: certifi 2025.11.12━━━━━━━━━━━ 1/8 [urllib3]
    Uninstalling certifi-2025.11.12:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [urllib3]
      Successfully uninstalled certifi-2025.11.12━━━━━━━━━━━━━━━━━ 3/8 [certifi]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [selenium]7/8 [selenium]ocket]


In [5]:
!pip install selenium webdriver-manager


In [45]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [79]:
options = webdriver.FirefoxOptions()
options.add_argument("--headless")

service = Service(GeckoDriverManager().install())

driver = webdriver.Firefox(service=service,options=options)

print("Browser started")

Browser started


In [80]:
wait = WebDriverWait(driver, 2)
url = "https://www.scrapingcourse.com/infinite-scrolling"

driver.get(url)
wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".product-item"))
)

print("Website opened")

Website opened


In [81]:
all_products = {}

def collect_products(batch_num):
    cards = driver.find_elements(By.CSS_SELECTOR, ".product-item")

    for card in cards:
        try:
            name = card.find_element(By.CSS_SELECTOR, ".product-name").text.strip()
            price = card.find_element(By.CSS_SELECTOR, ".product-price").text.strip()
            image_url = card.find_element(By.CSS_SELECTOR, "img.product-image").get_attribute("src")
            detail_url = card.find_element(By.CSS_SELECTOR, "a").get_attribute("href")

            if not detail_url:
                continue

            if detail_url not in all_products:
                all_products[detail_url] = {
                    "Product name": name,
                    "Price": price,
                    "Image URL": image_url,
                    "Scroll batch": batch_num,
                    "Detail URL": detail_url
                }

        except Exception:
            print("Error:", e)
            
collect_products(0)
print("products that are already placed on the website before any scroll:", len(all_products))


products that are already placed on the website before any scroll: 12


In [82]:
batch = 0

while True:
    old_count = len(driver.find_elements(By.CLASS_NAME, "product-item"))

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    try:
        wait.until(lambda d: len(d.find_elements(By.CLASS_NAME, "product-item")) > old_count)
    except Exception:
        print("No more products are loading.")
        break

    batch += 1
    collect_products(batch)

    new_count = len(driver.find_elements(By.CLASS_NAME, "product-item"))

    print("Batch:", batch,"- Unique products:", len(all_products))

    if new_count == old_count:
        print("No more products loading.")
        break

print("Total unique products found:", len(all_products))

Batch: 1 - Unique products: 12
Batch: 2 - Unique products: 22
Batch: 3 - Unique products: 32
Batch: 4 - Unique products: 42
Batch: 5 - Unique products: 52
Batch: 6 - Unique products: 62
Batch: 7 - Unique products: 72
Batch: 8 - Unique products: 82
Batch: 9 - Unique products: 92
Batch: 10 - Unique products: 102
Batch: 11 - Unique products: 112
Batch: 12 - Unique products: 122
Batch: 13 - Unique products: 132
Batch: 14 - Unique products: 142
Batch: 15 - Unique products: 147
No more products are loading.
Total unique products found: 147


In [83]:
driver.quit()

len(all_products)

for url, product in all_products.items():
    
    print(product["Product name"],"-> Batch",product["Scroll batch"])

Chaz Kangeroo Hoodie -> Batch 0
Teton Pullover Hoodie -> Batch 0
Bruno Compete Hoodie -> Batch 0
Frankie Sweatshirt -> Batch 0
Hollister Backyard Sweatshirt -> Batch 0
Stark Fundamental Hoodie -> Batch 0
Hero Hoodie -> Batch 0
Oslo Trek Hoodie -> Batch 0
Abominable Hoodie -> Batch 0
Mach Street Sweatshirt -> Batch 0
Grayson Crewneck Sweatshirt -> Batch 0
Ajax Full-Zip Sweatshirt -> Batch 0
Marco Lightweight Active Hoodie -> Batch 2
Beaumont Summit Kit -> Batch 2
Hyperion Elements Jacket -> Batch 2
Montana Wind Jacket -> Batch 2
Kenobi Trail Jacket -> Batch 2
Jupiter All-Weather Trainer -> Batch 2
Orion Two-Tone Fitted Jacket -> Batch 2
Lando Gym Jacket -> Batch 2
Taurus Elements Shell -> Batch 2
Mars HeatTech&trade; Pullover -> Batch 2
Typhon Performance Fleece-lined Jacket -> Batch 3
Proteus Fitness Jackshirt -> Batch 3
Caesar Warm-Up Pant -> Batch 3
Viktor LumaTech&trade; Pant -> Batch 3
Geo Insulated Jogging Pant -> Batch 3
Supernova Sport Pant -> Batch 3
Kratos Gym Pant -> Batch 3


In [84]:
product_details = []

for detail_url, product in all_products.items():

    print("Getting tthe details of :", product["Product name"])

    try:
        response = requests.get(detail_url,timeout=10)

        response.raise_for_status()

        soup = BeautifulSoup(response.text,"html.parser")

        #sku
        sku_element = soup.find("span",class_="sku")

        if sku_element:
            sku = sku_element.get_text(strip=True)
        else:
            sku = ""

        #descrip
        description_element = soup.find(
            "div",
            class_="woocommerce-product-details__short-description"
        )

        if description_element:
            description = description_element.get_text(" ",strip=True)
        else:
            description = ""

    except Exception as e:
        print("Error:", e)
        sku = "N/A"
        description = "N/A"

    product_details.append({
        "Product name": product["Product name"],
        "Price": product["Price"],
        "Image URL": product["Image URL"],
        "Scroll batch": product["Scroll batch"],
        "SKU": sku,
        "Short description": description,
        "Detail URL": detail_url
    })


print("\nDetail page scraping Done")
print("Total products:", len(product_details))

Getting tthe details of : Chaz Kangeroo Hoodie
Getting tthe details of : Teton Pullover Hoodie
Getting tthe details of : Bruno Compete Hoodie
Getting tthe details of : Frankie Sweatshirt
Getting tthe details of : Hollister Backyard Sweatshirt
Getting tthe details of : Stark Fundamental Hoodie
Getting tthe details of : Hero Hoodie
Getting tthe details of : Oslo Trek Hoodie
Getting tthe details of : Abominable Hoodie
Getting tthe details of : Mach Street Sweatshirt
Getting tthe details of : Grayson Crewneck Sweatshirt
Getting tthe details of : Ajax Full-Zip Sweatshirt
Getting tthe details of : Marco Lightweight Active Hoodie
Getting tthe details of : Beaumont Summit Kit
Getting tthe details of : Hyperion Elements Jacket
Getting tthe details of : Montana Wind Jacket
Getting tthe details of : Kenobi Trail Jacket
Getting tthe details of : Jupiter All-Weather Trainer
Getting tthe details of : Orion Two-Tone Fitted Jacket
Getting tthe details of : Lando Gym Jacket
Getting tthe details of : Ta

reason of using request rather than selenium for details page.

we used request rather than using selenium because selenium is used to scroll on the catalog page that is dynamic, and as the details pages are static, so using request for them is better or else if we use selenium for them then the scroll position of the catalog page will be lost and have to burdenize the program and retry everything to reach that certain scroll each time for every card we open.

In [85]:
df = pd.DataFrame(product_details)

df.to_csv("23L_0570_versionA_dynamic_products.csv",index=False)
print("CSV saved:23L_0570_versionA_dynamic_products.csv")

CSV saved:23L_0570_versionA_dynamic_products.csv


Scraping Methodology

How you identified the relevant elements/records on each page ?

answer:by inspecting each product card and finding what hierarchy or pattern each card has similar

How you navigated across pages (pagination, scrolling, clicking, etc.).

Answer: i used explicit waits rather than sleep because it will scrap only when actual product appears dynamically.

How you handled dynamic content in Question 2?

Answer: I stored each product by its detail-page URL and recorded the scroll batch in which it first appeared, which is important proof that the dynamic loading was working.

How your program decided that scraping was complete?

answer: each scroll compares the old quantity with new quantity and when it comes same then the code knows that no new products so it stops then. 

Any challenges you ran into and how you resolved them?

answer: while doing static and dynamic side by side, using selenium for category page and use of request for details page, i understood this thing with the help of ai because this doing side by side was new to me a bit.

How you verified that your scraper was actually collecting correct, complete data (spot checks,counts, etc.)?

Answer: ok so i checked each page and count of product on each page and then multiplied them and also dod spot checking that if that certain product is in that certain page and checked that batch 0 and batch 1 products are same so this helped me to check that my duplication check is working perfectly.

Validation Statistics

Total scroll batches processed: 15

Total unique products discovered: 147

Total products extracted with all required fields: 147 

Products skipped or failed: none

Products found before any scrolling: 12 before started scrolling

Products found only after scrolling: 147 (batch 1 products were duplicates)


github link : https://github.com/atikaahussain/Web-Scraping-DataScience-Assignment